In [1]:
import os
import subprocess

# Verifica o Python está vendo como JAVA_HOME
print(f"JAVA_HOME atual: {os.environ.get('JAVA_HOME')}")

# Verifica a versão real que o sistema responde
try:
    java_version = subprocess.check_output(['java', '-version'], stderr=subprocess.STDOUT).decode()
    print(f"Versão do Java no sistema:\n{java_version}")
except Exception as e:
    print(f"Erro ao verificar java: {e}")

JAVA_HOME atual: /usr/lib/jvm/java-17-openjdk-amd64
Versão do Java no sistema:
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-124.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-124.04.1, mixed mode, sharing)



In [2]:
from pyspark.sql import SparkSession
from delta import *
import os

# Configurando a sessão com suporte ao Delta Lake
builder = SparkSession.builder.appName("Trabalho_Engenharia_Dados") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0") \
    .config("spark.sql.warehouse.dir", "spark-warehouse")

# Inicializa a sessão Spark
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print(" Ambiente Delta Lake pronto e configurado!")

26/04/28 01:22:36 WARN Utils: Your hostname, MSI resolves to a loopback address: 127.0.1.1; using 172.25.175.71 instead (on interface eth0)
26/04/28 01:22:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/elison/trabalho_eng_dados/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/elison/.ivy2/cache
The jars for the packages stored in: /home/elison/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a826b55f-ca24-440d-acde-9abadfc09a14;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 275ms :: artifacts dl 14ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0

 Ambiente Delta Lake pronto e configurado!


In [3]:
# 1. Dados de carros
dados_carros = [
    (1, "Toyota", "Corolla", 2020),
    (2, "Honda", "Civic", 2019),
    (3, "Ford", "Focus", 2018)
]
colunas_carros = ["id", "marca", "modelo", "ano"]

# 2. Criando o DataFrame
df_carros = spark.createDataFrame(dados_carros, colunas_carros)


df_carros.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("../tabela_delta_trabalho")

print(" dados de carros  salvos.")

# 4. Visualizando
spark.read.format("delta").load("../tabela_delta_trabalho").show()

 dados de carros  salvos.


+---+------+-------+----+
| id| marca| modelo| ano|
+---+------+-------+----+
|  1|Toyota|Corolla|2020|
|  2| Honda|  Civic|2019|
|  3|  Ford|  Focus|2018|
+---+------+-------+----+



In [4]:
from delta.tables import *

dt = DeltaTable.forPath(spark, "tabela_delta_trabalho")

# 1. Mudar o modelo do Toyota (Update)

dt.update(
    condition="id = 1", 
    set={"modelo": "'Corolla Híbrido'"}
)

# 2. Deletar o carro da Ford (Delete)
# 
dt.delete("id = 3")

print(" Dados  modificados com sucesso!")

# Mostrar o resultado final
dt.toDF().show()

 Dados  modificados com sucesso!


+---+------+---------------+----+
| id| marca|         modelo| ano|
+---+------+---------------+----+
|  1|Toyota|Corolla Híbrido|2020|
|  2| Honda|          Civic|2019|
+---+------+---------------+----+



----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 55964)
Traceback (most recent call last):
  File "/usr/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.12/socketserver.py", line 761, in __init__
    self.handle()
  File "/home/elison/trabalho_eng_dados/.venv/lib/python3.12/site-packages/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/home/elison/trabalho_eng_dados/.venv/lib/python3.12/site-packages/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/home/elison/trabalho_eng_dados/

In [ ]:
# 1. Ver o histórico completo desde o teste
print(" Histórico de Alterações da Tabela:")
dt.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

# 2. Ver como a tabela era ANTES de deletar o Ford e mudar o Toyota
versao_anterior = 1 
df_passado = spark.read.format("delta").option("versionAsOf", versao_anterior).load("tabela_delta_trabalho")

print(f" Dados da Versão {versao_anterior} (Antes do Update/Delete):")
df_passado.show()

# 3. Histórico da primeira tabela :
print(" Dados da Versão 0 (Quando eram Funcionários):")
spark.read.format("delta").option("versionAsOf", 0).load("tabela_delta_trabalho").show()